## LSA(잠재 의미 분석)
- 문서 안에서 단어 사이의 잠재적인 의미 구조를 추출하는 기법
- TF-IDF 방식은 단어 간의 의미적 유사성을 반영 X
- TF-IDF에서 SVD 분해를 하여 단어 간의 의미를 파악
- TF-IDF에서 차원 축소(PCA, t-SNE와 같은 축소 기법 중 하나를 사용)하여 관계성을 확인
- LSA 효과
    - 벡터 공간의 차원을 줄여서 계산 효율을 증가 (차원 축소)
    - '영화', '필름' 비슷한 문맥의 단어를 가까운 벡터로 이동 (의미 유추)
    - 문서 주제별로 분류 기능 존재 (토픽 분석)

- TruncatedSVD (차원 축소 모델)
    - 절단된 특이값의 분해
    - 고차원의 희소 행렬(값이 0인 행렬)을 낮은 차원으로 압출하여 데이터 구조적 의미를 유지
    - 자연어 처리, 추천 시스템, 의미 분석, 잠재적인 토픽 분석 주로 사용

    - TF-IDF 행렬은 우선 고차원 -> 저차원
    - 0으로 이루어진 희소행렬들을 구조적인 의미를 유지하면서 값들을 부여
    - 같은 토픽의 문서는 같은 벡터 공간에서 가깝게 위치 -> 유사도 기반 자연어 처리에 활용

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd
from konlpy.tag import Okt

In [2]:
# 토큰화 함수를 정의 -> pos 필터
okt = Okt()

def tokenize(text):
    result= [word for word, pos in okt.pos(text)
             if pos in ['Noun', 'Adjective', 'Verb']]
    return result

In [3]:
docs = [
    '이 영화 정말 재미있었다',
    '매우 연기가 뛰어나다',
    '이 영화 별로다',
    '지루한 영화는 보기 어렵다',
    '정말 훌륭한 연기였다',
    '연기가 별로라서 지루했다'
]

In [4]:
# TF-IDF 벡터화
tfidf = TfidfVectorizer(
    tokenizer = tokenize,
    ngram_range=(1, 1),
    min_df = 1,
    max_df = 0.8,
    sublinear_tf=True,
    lowercase=False
)

In [5]:
X_tfidf = tfidf.fit_transform(docs)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [6]:
X_tfidf.shape

(6, 14)

In [8]:
lsa = TruncatedSVD(n_components=2, random_state=42)
X_lsa = lsa.fit_transform(X_tfidf)

In [9]:
df_lsa = pd.DataFrame(X_lsa, columns = ['topic1', 'topic2'])
df_lsa

,topic1,topic2
0,0.722615,-0.336336
1,0.229990,0.678032
2,0.801574,-0.279071
3,0.346712,-0.383244
4,0.392081,0.481308
5,0.516723,0.493418


In [10]:
df_lsa['document'] = docs
df_lsa

,topic1,topic2,document
0,0.722615,-0.336336,이 영화 정말 재미있었다
1,0.229990,0.678032,매우 연기가 뛰어나다
2,0.801574,-0.279071,이 영화 별로다
3,0.346712,-0.383244,지루한 영화는 보기 어렵다
4,0.392081,0.481308,정말 훌륭한 연기였다
5,0.516723,0.493418,연기가 별로라서 지루했다


In [13]:
terms = tfidf.get_feature_names_out()
components = lsa.components_

df_terms = pd.DataFrame(components.T, index = terms,
                        columns = ['topic1', 'topic2'])
df_terms

,topic1,topic2
뛰어나다,0.083061,0.338339
매우,0.083061,0.338339
별로,0.441011,0.083596
보기,0.105700,-0.161434
어렵다,0.105700,-0.161434
연기,0.283132,0.564684
였다,0.125589,0.213017
영화,0.476112,-0.333026
이,0.477259,-0.262077
재미있었다,0.244520,-0.157252


1. ratings_test.txt 파일 로드
2. 결측치 제외, id 컬럼 제외
3. document 의 중복된 데이터를 제거
4. 상위 5000개를 필터
5. 독립변수, 종속변수 train, test로 분할
6. X_train을 이용하여 tdifd, LSA 작업
7. 모델은 SVC 사용하여 학습 및 평가

In [14]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [20]:
df = pd.read_csv('../data_git/data_NLP/ratings_test.txt', sep = '\t')

In [21]:
df.dropna(inplace=True)
df.drop('id', axis=1, inplace=True)
df


,document,label
0,굳 ㅋ,1
1,GDNTOPCLASSINTHECLUB,0
2,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...
49995,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


In [23]:
# document 컬럼의 데이터 중 중복 데이터 제거
df.drop_duplicates('document', inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49157 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   document  49157 non-null  object
 1   label     49157 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.1+ MB


In [24]:
# 독립, 종속 변수 생성
X = df['document'].values
Y = df['label'].values

In [25]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

In [37]:
# 빠른 실행을 위해 데이터 개수 줄여준다
X_train = X_train[:5000]
X_test = X_test[:1000]
Y_train = Y_train[:5000]
Y_test = Y_test[:1000]

In [28]:
okt = Okt()
def tokenize(text):
    return okt.morphs(text)

# 벡터화
vector = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=3
)

In [29]:
# train 데이터를 이용하여 벡터화 작업
vector.fit(X_train)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...t 0x12a4a6e80>
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [32]:
X_tr_vc = vector.transform(X_train)
# train 데이터로 학습한 변환 모델에 test 데이터로 변환
X_te_vc = vector.transform(X_test)

In [34]:
svc = SVC(kernel = 'linear', C=1.0, random_state=42)
svc.fit(X_tr_vc, Y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [35]:
# test 데이터를 이용하여 예측
pred_vc = svc.predict(X_te_vc)


In [42]:
acc_vc = accuracy_score(pred_vc, Y_test)
f1_vc = f1_score(pred_vc, Y_test)
print(f"TD-IDF 변환 후 학습 \n 정확도 : {round(acc_vc, 4)} \n f1 : {round(f1_vc, 4)}")


TD-IDF 변환 후 학습 
 정확도 : 0.796 
 f1 : 0.7914


In [59]:
# LSA 정의
lsa = TruncatedSVD(
    n_components=200,
    random_state=42
)
# lsa를 이용한 학습
X_tr_lsa = lsa.fit_transform(X_tr_vc)
X_te_lsa = lsa.transform(X_te_vc)

In [60]:
svc.fit(X_tr_lsa, Y_train)
pred_lsa = svc.predict(X_te_lsa)

In [61]:
acc_lsa = accuracy_score(pred_lsa, Y_test)
f1_lsa = f1_score(pred_lsa, Y_test)
print(f"TD-IDF 변환 후 lsa 후 학습 \n 정확도 : {round(acc_lsa, 4)} \n f1 score : {round(f1_lsa, 4)}")

TD-IDF 변환 후 lsa 후 학습 
 정확도 : 0.744 
 f1 score : 0.7294


In [62]:
from sklearn.preprocessing import StandardScaler

In [65]:
scaler = StandardScaler(with_mean=True)

In [66]:
X_tr_std = scaler.fit_transform(X_tr_lsa)
X_te_std = scaler.transform(X_te_lsa)

In [67]:
svc.fit(X_tr_std, Y_train)
pred_scaler = svc.predict(X_te_std)

In [69]:
acc_std = accuracy_score(pred_scaler, Y_test)
f1_std = f1_score(pred_scaler, Y_test)

print(f"TD-IDF -> LSA -> Scaler 작업 후 정확도 : {round(acc_std, 4)}")
print(f"TD-IDF -> LSA -> Scaler 작업 후 f1 : {round(f1_std, 4)}")

TD-IDF -> LSA -> Scaler 작업 후 정확도 : 0.745
TD-IDF -> LSA -> Scaler 작업 후 f1 : 0.7341
